# Saudi Digital Concierge — Data Exploration
## Source 5: Enjoy.sa Events API

**Endpoint:** `https://enjoy.sa/api/v1/odp/events/Get`
**Purpose:** events, dates, times, city, category, and family/gender constraints.

This notebook fetches the API and answers 12 specific questions about the data. It uses
**defensive schema discovery** — column names are located by keyword, so the analysis
adapts to whatever fields the API returns (English or Arabic).

> **Run this on a machine with internet access to `enjoy.sa`.** If the endpoint needs
> query params, an API key, or a specific host header, adjust `PARAMS`/`HEADERS` below.

### Questions answered
1. Number of events · 2. Available fields · 3. Date coverage · 4. City coverage ·
5. Event categories/types · 6. Start/end dates · 7. Start/end times ·
8. Family/male/female restrictions · 9. Missing values · 10. Duplicate events ·
11. Active/future events? · 12. Pagination / record limits

In [24]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
%pip install requests
import requests
from datetime import datetime, timezone

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

URL = "https://enjoy.sa/api/v1/odp/events/Get"
HEADERS = {"Accept": "application/json", "User-Agent": "saudi-digital-concierge/1.0"}
PARAMS = {}          # add query params here if the API requires them
TIMEOUT = 60
from datetime import datetime, timezone

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

URL = "https://enjoy.sa/api/v1/odp/events/Get"
HEADERS = {"Accept": "application/json", "User-Agent": "saudi-digital-concierge/1.0"}
PARAMS = {}          # add query params here if the API requires them
TIMEOUT = 60

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0. Fetch the raw response

We first inspect the *shape* of the response — a bare JSON array, or an object that wraps
the records (e.g. `{"data": [...]}`, `{"items": [...]}`, `{"result": {"events": [...]}}`)
and may carry paging metadata.

In [25]:
resp = requests.get(URL, headers=HEADERS, params=PARAMS, timeout=TIMEOUT)
print("HTTP", resp.status_code, "| content-type:", resp.headers.get("content-type"))
resp.raise_for_status()
payload = resp.json()

print("Top-level type:", type(payload).__name__)
if isinstance(payload, dict):
    print("Top-level keys:", list(payload.keys()))

HTTP 200 | content-type: application/json; charset=utf-8
Top-level type: dict
Top-level keys: ['Status', 'IsSucceeded', 'Data', 'Message']


In [26]:
def extract_records(payload):
    """Return the list of event records from whatever envelope the API uses."""
    if isinstance(payload, list):
        return payload, None
    if isinstance(payload, dict):
        # common list-bearing keys, checked in order
        for key in ["data", "items", "results", "result", "events",
                    "records", "value", "Data", "Items", "Result", "Events"]:
            if key in payload and isinstance(payload[key], list):
                return payload[key], payload
        # one level deeper (e.g. {"result": {"events": [...]}})
        for v in payload.values():
            if isinstance(v, dict):
                for key, inner in v.items():
                    if isinstance(inner, list):
                        return inner, payload
        # otherwise the dict itself may be a single record
        return [payload], payload
    return [], payload

records, envelope = extract_records(payload)
print("Extracted", len(records), "record(s).")
if records:
    print("First record keys:", list(records[0].keys()) if isinstance(records[0], dict) else type(records[0]))

Extracted 5802 record(s).
First record keys: ['Name', 'City', 'StartDate', 'StartDateFormatted', 'EndDate', 'EndDateFormatted', 'StartTime', 'StartTimeFormatted', 'EndTime', 'EndTimeFormatted', 'IsMaleAllowed', 'IsFemaleAllowed', 'IsFamilyAllowed', 'EventMode']


In [27]:
# Flatten nested JSON into a tidy table.
df = pd.json_normalize(records)
print("DataFrame shape:", df.shape)
df.head(3)

DataFrame shape: (5802, 14)


,Name,City,StartDate,StartDateFormatted,EndDate,EndDateFormatted,StartTime,StartTimeFormatted,EndTime,EndTimeFormatted,IsMaleAllowed,IsFemaleAllowed,IsFamilyAllowed,EventMode
0,الألعاب النارية,NaN,2018-09-23T00:00:00Z,23-09-2018,2018-09-23T00:00:00Z,23-09-2018,22:00:00,10:00 م,22:30:00,10:30 م,True,True,True,IsExpired
1,الألعاب النارية,NaN,2018-09-23T00:00:00Z,23-09-2018,2018-09-23T00:00:00Z,23-09-2018,22:00:00,10:00 م,22:30:00,10:30 م,True,True,True,IsExpired
2,الألعاب النارية,NaN,2018-09-23T00:00:00Z,23-09-2018,2018-09-23T00:00:00Z,23-09-2018,22:00:00,10:00 م,22:30:00,10:30 م,True,True,True,IsExpired


## 1. Number of events

In [28]:
print("Number of records returned:", len(df))
# NOTE: this is the count in THIS response; see Q12 (pagination) for the true total.

Number of records returned: 5802


## 2. Available fields

In [29]:
fields = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "non_null": df.notna().sum().values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
fields

,column,dtype,non_null,n_unique,example
0,Name,str,5802,4399,الألعاب النارية
1,City,str,5798,74,جدة
2,StartDate,str,5802,1949,2018-09-23T00:00:00Z
3,StartDateFormatted,str,5802,1832,23-09-2018
4,EndDate,str,5802,2020,2018-09-23T00:00:00Z
5,EndDateFormatted,str,5802,1913,23-09-2018
6,StartTime,str,5802,86,22:00:00
7,StartTimeFormatted,str,5802,86,10:00 م
8,EndTime,str,5802,101,22:30:00
9,EndTimeFormatted,str,5802,101,10:30 م


In [30]:
# Helper: locate columns whose name contains any of the given keywords.
def find_cols(keywords):
    kws = [k.lower() for k in keywords]
    return [c for c in df.columns if any(k in c.lower() for k in kws)]

col_map = {
    "city":      find_cols(["city", "region", "location", "المدينة", "مدينه", "منطقة"]),
    "category":  find_cols(["category", "categories", "type", "tag", "التصنيف", "نوع", "فئة"]),
    "start":     find_cols(["start", "from", "begin", "بداية", "من"]),
    "end":       find_cols(["end", "to", "finish", "نهاية", "الى", "إلى"]),
    "date":      find_cols(["date", "day", "تاريخ", "يوم"]),
    "time":      find_cols(["time", "hour", "وقت", "ساعة"]),
    "gender":    find_cols(["gender", "male", "female", "family", "audience",
                            "جنس", "رجال", "نساء", "عائلات", "عائلي"]),
    "name":      find_cols(["name", "title", "اسم", "عنوان"]),
}
for k, v in col_map.items():
    print(f"{k:10s}: {v}")

city      : ['City']
category  : []
start     : ['StartDate', 'StartDateFormatted', 'StartTime', 'StartTimeFormatted']
end       : ['EndDate', 'EndDateFormatted', 'EndTime', 'EndTimeFormatted']
date      : ['StartDate', 'StartDateFormatted', 'EndDate', 'EndDateFormatted']
time      : ['StartTime', 'StartTimeFormatted', 'EndTime', 'EndTimeFormatted']
gender    : ['IsMaleAllowed', 'IsFemaleAllowed', 'IsFamilyAllowed']
name      : ['Name']


## 3. Date coverage
The earliest and latest dates present, across all detected date columns.

In [31]:
date_cols = sorted(set(col_map["date"] + col_map["start"] + col_map["end"]))
print("Date-like columns:", date_cols)

parsed = {}
for col in date_cols:
    s = pd.to_datetime(df[col], errors="coerce", utc=True)
    if s.notna().any():
        parsed[col] = s
        print(f"  {col:30s} min={s.min()}  max={s.max()}  parsed={s.notna().sum()}/{len(s)}")

if parsed:
    all_dates = pd.concat(parsed.values())
    print("\nOVERALL date coverage:", all_dates.min(), "->", all_dates.max())
else:
    print("No parseable date columns found — inspect the fields table in Q2.")

Date-like columns: ['EndDate', 'EndDateFormatted', 'EndTime', 'EndTimeFormatted', 'StartDate', 'StartDateFormatted', 'StartTime', 'StartTimeFormatted']
  EndDate                        min=2017-01-06 00:00:00+00:00  max=2026-07-22 00:00:00+00:00  parsed=2831/5802
  EndDateFormatted               min=2017-01-06 00:00:00+00:00  max=2031-05-04 00:00:00+00:00  parsed=5802/5802
  EndTime                        min=2026-09-10 00:00:00+00:00  max=2026-09-10 23:59:00+00:00  parsed=5802/5802
  StartDate                      min=2017-01-05 00:00:00+00:00  max=2026-02-09 00:00:00+00:00  parsed=2831/5802
  StartDateFormatted             min=2017-01-05 00:00:00+00:00  max=2026-10-01 00:00:00+00:00  parsed=5802/5802
  StartTime                      min=2026-09-10 00:00:00+00:00  max=2026-09-10 23:07:00+00:00  parsed=5802/5802

OVERALL date coverage: 2017-01-05 00:00:00+00:00 -> 2031-05-04 00:00:00+00:00


C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\3326609540.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\3326609540.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\3326609540.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\332660954

## 4. City coverage

In [32]:
if col_map["city"]:
    for col in col_map["city"]:
        vc = df[col].value_counts(dropna=False)
        print(f"=== {col} — {df[col].nunique(dropna=True)} unique ===")
        print(vc.head(30).to_string(), "\n")
else:
    print("No city/location column detected — check Q2 fields.")

=== City — 74 unique ===
City
الرياض              3012
جدة                 1054
الطائف               214
الخبر                157
الدمام               141
كل مناطق المملكة     120
المدينة المنورة      105
الأحساء              103
أبها                  99
الخرج                 80
جازان                 60
الباحة                56
تبوك                  54
حائل                  53
الظهران               49
بريدة                 43
الجبيل                35
الدرعية               30
مكة                   29
نجران                 27
عرعر                  26
عنيزة                 23
سكاكا                 21
القصيم                21
ينبع                  20
حفر الباطن            19
القطيف                18
الجوف                 12
شقراء                  9
رابغ                   9 



## 5. Event categories / types

In [41]:
if col_map["category"]:
    for col in col_map["category"]:
        print(f"=== {col} — {df[col].nunique(dropna=True)} unique ===")
        # categories may be lists; explode if so
        sample = df[col].dropna()
        if len(sample) and isinstance(sample.iloc[0], list):
            print(df[col].explode().value_counts().head(30).to_string(), "\n")
        else:
            print(df[col].value_counts(dropna=False).head(30).to_string(), "\n")
else:
    print("No category/type column detected — check Q2 fields.")

No category/type column detected — check Q2 fields.


## 6. Start / end dates &nbsp;·&nbsp; 7. Start / end times

In [34]:
start_cols = col_map["start"]
end_cols   = col_map["end"]
print("Start columns:", start_cols)
print("End columns:  ", end_cols)

def summarise_datetime(col):
    s = pd.to_datetime(df[col], errors="coerce", utc=True)
    if s.notna().any():
        print(f"  {col}: {s.min()} -> {s.max()} "
              f"(parsed {s.notna().sum()}/{len(s)})")
        # time-of-day distribution
        hours = s.dt.hour.dropna()
        if len(hours):
            print(f"      hour-of-day range: {int(hours.min())}h - {int(hours.max())}h")

print("\n-- Start --")
for col in start_cols: summarise_datetime(col)
print("-- End --")
for col in end_cols: summarise_datetime(col)

# Dedicated time columns (if separate from dates)
time_only = [c for c in col_map["time"] if c not in start_cols + end_cols]
if time_only:
    print("\n-- Standalone time columns --")
    for col in time_only:
        print(f"  {col}: sample ->", df[col].dropna().head(5).tolist())

Start columns: ['StartDate', 'StartDateFormatted', 'StartTime', 'StartTimeFormatted']
End columns:   ['EndDate', 'EndDateFormatted', 'EndTime', 'EndTimeFormatted']

-- Start --
  StartDate: 2017-01-05 00:00:00+00:00 -> 2026-02-09 00:00:00+00:00 (parsed 2831/5802)
      hour-of-day range: 0h - 19h


  StartDateFormatted: 2017-01-05 00:00:00+00:00 -> 2026-10-01 00:00:00+00:00 (parsed 5802/5802)
      hour-of-day range: 0h - 0h
  StartTime: 2026-09-10 00:00:00+00:00 -> 2026-09-10 23:07:00+00:00 (parsed 5802/5802)
      hour-of-day range: 0h - 23h
-- End --
  EndDate: 2017-01-06 00:00:00+00:00 -> 2026-07-22 00:00:00+00:00 (parsed 2831/5802)
      hour-of-day range: 0h - 0h
  EndDateFormatted: 2017-01-06 00:00:00+00:00 -> 2031-05-04 00:00:00+00:00 (parsed 5802/5802)
      hour-of-day range: 0h - 0h
  EndTime: 2026-09-10 00:00:00+00:00 -> 2026-09-10 23:59:00+00:00 (parsed 5802/5802)
      hour-of-day range: 0h - 23h


C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\1649650976.py:7: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\1649650976.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\1649650976.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(df[col], errors="coerce", utc=True)
C:\Users\bedoor.alsulami\AppData\Local\Temp\ipykernel_34860\164965097

## 8. Family / male / female restrictions
Audience-restriction fields tell the concierge who an event is open to.

In [35]:
if col_map["gender"]:
    for col in col_map["gender"]:
        print(f"=== {col} ===")
        sample = df[col].dropna()
        if len(sample) and isinstance(sample.iloc[0], list):
            print(df[col].explode().value_counts(dropna=False).to_string(), "\n")
        else:
            print(df[col].value_counts(dropna=False).to_string(), "\n")
else:
    print("No explicit gender/family/audience column detected.")
    print("Scan the fields table in Q2 for values like 'Family', 'Men', 'Women', "
          "'عائلات', 'رجال', 'نساء'.")

=== IsMaleAllowed ===
IsMaleAllowed
True     5290
False     471
None       41 

=== IsFemaleAllowed ===
IsFemaleAllowed
True     5345
False     416
None       41 

=== IsFamilyAllowed ===
IsFamilyAllowed
True     5240
False     521
None       41 



## 9. Missing values

In [36]:
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
}).sort_values("missing", ascending=False)
print("Columns with any missing values:")
miss[miss["missing"] > 0]

Columns with any missing values:


,missing,missing_%
IsFamilyAllowed,41,0.7
IsFemaleAllowed,41,0.7
IsMaleAllowed,41,0.7
City,4,0.1


## 10. Duplicate events
Checked two ways: fully identical rows, and duplicates on a likely business key
(id, or name + start).

In [37]:
print("Fully identical rows:", df.duplicated().sum())

id_cols = find_cols(["id", "uuid", "guid", "code"])
id_cols = [c for c in id_cols if "video" not in c.lower() and "image" not in c.lower()]
print("Candidate id columns:", id_cols)
for col in id_cols[:2]:
    dups = df[col].duplicated().sum()
    print(f"  duplicate {col}: {dups}")

key = col_map["name"][:1] + col_map["start"][:1]
if len(key) >= 1:
    print(f"\nDuplicates on business key {key}:", df.duplicated(subset=key).sum())

Fully identical rows: 30
Candidate id columns: []

Duplicates on business key ['Name', 'StartDate']: 194


## 11. Active / future events?
Compares start/end dates to *now* to see whether the API serves upcoming events or only
a historical dump.

In [38]:
now = pd.Timestamp.now(tz="UTC")
print("Now:", now)

start_col = col_map["start"][0] if col_map["start"] else (date_cols[0] if date_cols else None)
end_col   = col_map["end"][0] if col_map["end"] else start_col

if start_col:
    s = pd.to_datetime(df[start_col], errors="coerce", utc=True)
    e = pd.to_datetime(df[end_col], errors="coerce", utc=True) if end_col else s
    future  = (s > now).sum()
    ongoing = ((s <= now) & (e >= now)).sum()
    past    = (e < now).sum()
    print(f"Using start='{start_col}', end='{end_col}'")
    print(f"  Future (not started yet): {future}")
    print(f"  Ongoing (active now):     {ongoing}")
    print(f"  Past (already ended):     {past}")
    print("\n=> API contains active/future events." if future + ongoing > 0
          else "\n=> API appears to contain only past events.")
else:
    print("No start-date column detected — cannot classify.")

Now: 2026-09-10 06:40:25.779752+00:00
Using start='StartDate', end='EndDate'
  Future (not started yet): 0
  Ongoing (active now):     0
  Past (already ended):     2831

=> API appears to contain only past events.


## 12. Pagination / record limits
Probe common paging schemes and compare returned counts. If the count changes with
`page`/`pageSize` (or the envelope carries a `total`), the endpoint is paginated.

In [39]:
# a) Paging metadata in the envelope?
if isinstance(envelope, dict):
    meta = {k: v for k, v in envelope.items()
            if any(t in k.lower() for t in ["total", "count", "page", "size", "limit", "next"])
            and not isinstance(v, (list, dict))}
    print("Envelope paging metadata:", meta if meta else "(none found)")

# b) Does the count respond to paging params?
def count_for(params):
    try:
        r = requests.get(URL, headers=HEADERS, params=params, timeout=TIMEOUT)
        recs, _ = extract_records(r.json())
        return len(recs)
    except Exception as ex:
        return f"error: {ex}"

base_n = len(df)
print("Baseline (no params):", base_n)
for params in [{"page": 1, "pageSize": 10}, {"page": 2, "pageSize": 10},
               {"pageNumber": 1, "pageSize": 10}, {"limit": 10, "offset": 0},
               {"limit": 10, "offset": 10}, {"skip": 0, "take": 10},
               {"skip": 10, "take": 10}, {"pageSize": 1000}]:
    print(f"  {params} -> {count_for(params)} records")

print("\nInterpretation:")
print("- If page=1 and page=2 return DIFFERENT records -> paginated; iterate to get all.")
print("- If pageSize=1000 returns more than baseline -> baseline was capped by a default limit.")
print("- If every request returns the same count -> no pagination (single full payload).")

Envelope paging metadata: (none found)
Baseline (no params): 5802


  {'page': 1, 'pageSize': 10} -> 10 records
  {'page': 2, 'pageSize': 10} -> 10 records
  {'pageNumber': 1, 'pageSize': 10} -> 10 records
  {'limit': 10, 'offset': 0} -> 5802 records
  {'limit': 10, 'offset': 10} -> 5802 records
  {'skip': 0, 'take': 10} -> 5802 records
  {'skip': 10, 'take': 10} -> 5802 records
  {'pageSize': 1000} -> 1000 records

Interpretation:
- If page=1 and page=2 return DIFFERENT records -> paginated; iterate to get all.
- If pageSize=1000 returns more than baseline -> baseline was capped by a default limit.
- If every request returns the same count -> no pagination (single full payload).


In [42]:
print(df[["StartDate", "StartDateFormatted",
          "EndDate", "EndDateFormatted"]].head(20).to_string())

               StartDate StartDateFormatted               EndDate EndDateFormatted
0   2018-09-23T00:00:00Z         23-09-2018  2018-09-23T00:00:00Z       23-09-2018
1   2018-09-23T00:00:00Z         23-09-2018  2018-09-23T00:00:00Z       23-09-2018
2   2018-09-23T00:00:00Z         23-09-2018  2018-09-23T00:00:00Z       23-09-2018
3   2018-09-23T00:00:00Z         23-09-2018  2018-09-23T00:00:00Z       23-09-2018
4    2026-10-01T00:00:00         01-10-2026   2026-10-01T00:00:00       01-10-2026
5    2026-09-18T00:00:00         18-09-2026   2026-09-18T00:00:00       18-09-2026
6    2026-09-17T00:00:00         17-09-2026   2026-09-17T00:00:00       17-09-2026
7    2026-09-04T00:00:00         04-09-2026   2026-09-04T00:00:00       04-09-2026
8    2026-09-03T00:00:00         03-09-2026   2026-09-03T00:00:00       03-09-2026
9    2026-08-27T00:00:00         27-08-2026   2026-08-27T00:00:00       27-08-2026
10   2026-08-14T00:00:00         14-08-2026   2026-08-14T00:00:00       14-08-2026
11  

In [43]:
print(df[["StartDate", "StartDateFormatted",
          "EndDate", "EndDateFormatted"]].tail(20).to_string())

                 StartDate StartDateFormatted               EndDate EndDateFormatted
5782   2024-09-19T00:00:00         19-09-2024   2024-09-21T00:00:00       21-09-2024
5783   2024-09-26T00:00:00         26-09-2024   2024-10-05T00:00:00       05-10-2024
5784  2019-09-19T00:00:00Z         19-09-2019  2019-09-28T00:00:00Z       28-09-2019
5785  2019-09-19T00:00:00Z         19-09-2019  2019-09-28T00:00:00Z       28-09-2019
5786  2020-09-08T00:00:00Z         08-09-2020  2020-09-08T00:00:00Z       08-09-2020
5787  2019-09-19T00:00:00Z         19-09-2019  2019-09-28T00:00:00Z       28-09-2019
5788  2019-09-19T00:00:00Z         19-09-2019  2019-09-28T00:00:00Z       28-09-2019
5789  2019-09-19T00:00:00Z         19-09-2019  2019-09-28T00:00:00Z       28-09-2019
5790  2020-01-03T00:00:00Z         03-01-2020  2020-01-19T00:00:00Z       19-01-2020
5791  2020-01-08T00:00:00Z         08-01-2020  2020-01-21T00:00:00Z       21-01-2020
5792   2023-09-29T00:00:00         29-09-2023   2023-09-29T00:00:

In [44]:
print(df["EventMode"].value_counts(dropna=False))

EventMode
IsExpired    5752
IsActive       50
Name: count, dtype: int64


In [45]:
active = df[df["EventMode"] == "IsActive"]

print(active.shape)

(50, 14)


In [48]:
cols = [
    "Name", "City", "StartDateFormatted",
    "EndDateFormatted", "StartTimeFormatted",
    "EndTimeFormatted", "IsMaleAllowed",
    "IsFemaleAllowed", "IsFamilyAllowed",
]

display(active[cols])


,Name,City,StartDateFormatted,EndDateFormatted,StartTimeFormatted,EndTimeFormatted,IsMaleAllowed,IsFemaleAllowed,IsFamilyAllowed
4,أمسية ليالي الخيمة الموسيقية,جدة,01-10-2026,01-10-2026,9:30 م,1:00 ص,True,True,True
5,بهاء سلطان ورامي جمال في جدة,جدة,18-09-2026,18-09-2026,8:00 م,12:00 ص,True,True,False
6,حفلة عمرو دياب في جدة,جدة,17-09-2026,17-09-2026,8:00 م,1:00 ص,True,True,False
15,جولة التحدي,جدة,29-12-2021,29-12-2030,5:00 م,11:45 م,True,True,True
24,فنكي منكي - جدة بارك,جدة,08-02-2024,08-02-2031,9:00 ص,12:00 ص,True,True,True
25,فنكي منكي - ردسي مول,جدة,27-02-2025,27-02-2031,9:00 ص,12:00 ص,True,True,True
26,مدينة بيلي بيز,جدة,13-03-2022,13-03-2031,9:00 ص,1:00 ص,True,True,True
27,مدينة بيلي بيز,جدة,13-03-2022,13-03-2031,9:00 ص,1:00 ص,True,True,True
34,اكستريم بلاي,جدة,23-07-2025,23-07-2030,3:00 م,12:00 ص,True,True,True
35,بوست,جدة,02-01-2023,02-01-2031,2:00 م,1:00 ص,True,True,True


## Summary of findings

Response envelope: `{ Status, IsSucceeded, Data, Message }` — event records under `Data`.

| # | Question | Finding |
|---|----------|---------|
| 1 | Number of events | **5,802** (full set returned by default) |
| 2 | Available fields | **14**: `Name`, `City`, `StartDate`/`StartDateFormatted`, `EndDate`/`EndDateFormatted`, `StartTime`/`StartTimeFormatted`, `EndTime`/`EndTimeFormatted`, `IsMaleAllowed`, `IsFemaleAllowed`, `IsFamilyAllowed`, `EventMode` |
| 3 | Date coverage | Event start dates **Jan 2017 → Oct 2026**; a few long-running/open-ended events carry later end dates (into ~2030) |
| 4 | City coverage | **74 cities** (Arabic). Top: الرياض 3,012 · جدة 1,054 · الطائف 214 · الخبر 157 · الدمام 141. Riyadh + Jeddah ≈ 70% |
| 5 | Categories / types | **None** — no category field. `EventMode` is a status flag, not a category |
| 6 | Start / end dates | Present for all rows. ISO (`StartDate`) and `*Formatted` (DD-MM-YYYY) are **consistent** — same dates in two formats |
| 7 | Start / end times | Present for all 5,802 (`HH:MM:SS` + Arabic 12h). 86 distinct start times, 101 end times |
| 8 | Family/male/female | Male 5,290✔/471�’/41 none · Female 5,345/416/41 · Family 5,240/521/41. Most events open to everyone |
| 9 | Missing values | Gender/family flags 41 each (0.7%); City 4 (0.1%). Dates/times complete |
| 10 | Duplicate events | **30** fully identical rows; **194** dupes on `Name`+`StartDate`. No unique ID field |
| 11 | Active / future events | **Yes — 50 `IsActive`** vs **5,752 `IsExpired`** (`EventMode`). Use `EventMode` for status, not date math |
| 12 | Pagination / limits | Default returns **all 5,802** (no cap). `page`+`pageSize` honoured (and `pageSize=1000`→1000); `limit/offset`, `skip/take` **ignored**; no paging metadata in envelope |

### Notes for cleaning (`src/cleaning`)
- **Dates are consistent** across the ISO and `*Formatted` columns. When parsing the
  `*Formatted` (DD-MM-YYYY) fields, pass `dayfirst=True` — otherwise a day > 12 is
  misread as a month and years get distorted (this produced a spurious "2031" in an
  earlier exploratory parse). Prefer the ISO `StartDate`/`EndDate` for machine parsing.
- **No unique ID** → dedupe on a composite key (`Name`+`City`+`StartDate`+`StartTime`);
  expect to drop ~30–194 rows.
- **Cities are Arabic strings** → map to the canonical city/province key used by the
  other sources (e.g. `الرياض`→Riyadh) for cross-source joins.
- **Filter with `EventMode`** (`IsActive` / `IsExpired`) to serve live events — 50 are
  currently active.
- **No categories** → if the concierge needs event types, derive them from `Name`
  (keyword/LLM tagging); the API does not provide them.

**Next:** save the raw pull to `data/raw/events/enjoy_events_<date>.json`, then implement
the cleaning above and write a tidy events table to `data/processed/`.